# Adding MC systematics (GENIE, Flux, G4, MCstat)

Covers adding rate/norm systematics via `funcs.py`'s `systs_input()`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
# TODO: set to the directory containing cafpybara/ (skipped if already importable).
sys.path.append("TODO_SET_ME/path/to/parent/of/cafpybara")

# TODO: set to your cafpyana clone (skipped if already exported).
os.environ.setdefault("CAFPYANA_PATH", "TODO_SET_ME/path/to/cafpyana")

import pandas as pd
import matplotlib.pyplot as plt
from cafpybara import core

# TODO: import your own analysis's name instead of _template.
import cafpybara.analyses._template as ana

STILL_ON_TEMPLATE = ana.__name__.endswith('_template')
if STILL_ON_TEMPLATE:
    print("Still importing _template directly -- see the TODO above.")
else:
    print("Imported cleanly -- a real, importable analysis.")

In [ ]:
# Define once, used throughout this notebook.
#
# MC_FILE needs real weights, e.g.
# `run_df_maker.py -c configs/hnl_mcnu_wgt.py -l <input.list> -o <output_name>`.
# GENIE/Flux/Geant4 weights included; cafpyana's own preselection runs too.
# DEFAULT_CUTS below is a separate, topology-specific cut on top.
#
# DATA_FILE/OFFBEAM_FILE use the same preselection-on config, e.g.
# `run_df_maker.py -c configs/hnl_data.py -l <input.list> -o <output_name>`.
#
# NOTE: run `concat_hdf.py` after producing per-shard dataframes.
# The resulting `file_idx` column is required to prevent duplication when computing `mcstat`.
MC_FILE = "your_mc_file.h5"
DATA_FILE = "your_data_file.h5"
OFFBEAM_FILE = "your_offbeam_file.h5"
VAR = "your_variable"
BINS = [0, 1, 2, 3]

## 1. Load MC + data

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    offbeam_df, offbeam_pot, offbeam_ngates = ana.load_data(
        OFFBEAM_FILE, onbeam=False, offbeam_signal_value=ana.signal_dict["offbeam"],
        cuts=ana.DEFAULT_CUTS, preprocess_fn=ana.preprocess.preprocess_data,
    )
    data_df, data_pot, data_ngates = ana.load_data(
        DATA_FILE, cuts=ana.DEFAULT_CUTS, preprocess_fn=ana.preprocess.preprocess_data,
    )
    mc_df, mc_pot, mc_ngen = ana.load_mc(
        MC_FILE, cuts=ana.DEFAULT_CUTS, preprocess_fn=ana.preprocess.preprocess_mc,
    )

    f = 0.0725  # TODO: double check for the most-up-to-date fudge factor
    ongates_per_pot = data_ngates / data_pot
    scale_offbeam = ((1 - f) * (ongates_per_pot * mc_pot)) / offbeam_ngates

    weight_col = ('weights_mc', '', '', '', '', '')
    mc_df[weight_col] = 1.0
    offbeam_df[weight_col] = scale_offbeam

    mcstat_cols = ['__ntuple', 'entry', 'rec.slc..index', 'run', 'subrun', 'evt', 'sample', 'file_idx']
    mc_df = pd.concat([
        ana.mcstat(mc_df.assign(sample=0).set_index("sample", append=True).reset_index(), cols=mcstat_cols),
        offbeam_df.assign(sample=1).set_index("sample", append=True).reset_index(),
    ])
    print("Merged MC + off-beam rows:", len(mc_df))

    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot)  # no systs= yet
except Exception as e:
    print("Expected -- neither a real data file nor an \"offbeam\" category exist yet:")
    print(f"  {type(e).__name__}: {e}")

## 2. `funcs.py` -- `systs_input()` (adding systematics)

* TODO: none new -- reuses `DEFAULT_CUTS`/`define_signal` from `analysis.py`.
* Arguments:
  * `mcbnb_pot` -- MC BNB POT to normalize to.
  * `cuts` -- defaults to `DEFAULT_CUTS` if omitted.
  * `uncertainty_keys` -- uncertainty sources to include (`'rate'`,
    `'xsec'`, `'detv'`, `'norm'`, `'cosmic'`).
* Returns: a `SystematicsInput`.
* Pass as `plot_var(..., systs=si)`/`plot_mc_data(..., systs=si)`, or
  unpack `si.to_kwargs()` into `get_total_cov`.
* Note: `'cosmic'` and `'detv'` are not included in this notebook yet.
  `'detv'` is added in `step4_add_detvar_systematics.ipynb`; `'cosmic'`
  needs an in-time-cosmic sample, not required by default.

See below for the actual call:

In [ ]:
print("ana.get_total_cov IS core.funcs.get_total_cov:", ana.get_total_cov is core.funcs.get_total_cov)
print("(if this were ever False for a new analysis, get_total_cov has been wrongly overridden)\n")

# What you'd actually write in your own analysis notebook:
try:
    si = ana.systs_input(mc_pot, uncertainty_keys={'rate', 'norm'})
    mc_syst_output = ana.get_total_cov(mc_df, reco_var=VAR, bins=BINS, **si.to_kwargs())
    print("mc_syst_output.rate_cov shape:", mc_syst_output.rate_cov.shape)
    print("mc_syst_output.rate_syst_dict keys:", list(mc_syst_output.rate_syst_dict.keys()))
except Exception as e:
    print("Expected -- neither a real data file nor an \"offbeam\" category exist yet:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# plot_mc_data with systematics on (reuses mc_syst_output from above):
try:
    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot, systs=mc_syst_output)
except Exception as e:
    print("Expected -- neither a real data file nor an \"offbeam\" category exist yet:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Systematics category breakdown
try:
    fig, axes, cats, cat_sums = ana.plot_syst_category_breakdown(
        syst_vars=[(mc_syst_output, BINS, VAR)],
        category_dict=ana.category_dict_signal,
    )
    plt.show()
except Exception as e:
    print("Expected -- neither a real data file nor an \"offbeam\" category exist yet:")
    print(f"  {type(e).__name__}: {e}")